In [ ]:
import pandas as pd
import numpy as np
import lightgbm as lgb
import optuna
from optuna.integration import LightGBMPruningCallback
from sklearn.metrics import roc_auc_score
import joblib
import time
import gc
import warnings

warnings.filterwarnings('ignore')
optuna.logging.set_verbosity(optuna.logging.WARNING)

print("✓ Imports ready")

✓ Imports ready


/opt/anaconda3/envs/customer_analytics/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
FEATURES = [
    'MONTH', 'DAY_OF_WEEK', 'DEP_HOUR', 'ARR_HOUR', 'IS_HOLIDAY',
    'DISTANCE', 'profit_margin', 'origin_monthly_passengers',
    'origin_temp', 'origin_dew_point', 'origin_pressure',
    'origin_wind_dir', 'origin_wind_speed', 'origin_precip_1hr',
    'origin_weather_severity',
    'dest_temp', 'dest_dew_point', 'dest_pressure',
    'dest_wind_dir', 'dest_wind_speed', 'dest_precip_1hr',
    'dest_weather_severity',
    'airline_delay_rate_30d', 'origin_delay_rate_30d', 'dest_delay_rate_30d',
    'route_delay_rate_30d', 'origin_avg_taxi_out_30d',
    'flight_num_delay_rate_30d', 'origin_hour_delay_rate_30d',
    'carrier_origin_delay_rate_30d', 'dest_hour_delay_rate_30d',
    'airline_delay_rate_7d', 'origin_delay_rate_7d',
    'cascade_score', 'cascade_delay_minutes', 'hours_since_last_delay',
    'hourly_flight_count', 'scheduled_turnaround_buffer', 'tail_flight_num_today',
    'dest_hourly_flight_count',
    'inbound_arr_delay_3h', 'dest_inbound_arr_delay_3h',
    'prev_tail_arr_delay', 'national_hub_delay_2h',
    'OP_UNIQUE_CARRIER', 'ORIGIN', 'DEST', 'airline_cluster_label',
]

CAT_FEATURES = ['OP_UNIQUE_CARRIER', 'ORIGIN', 'DEST', 'airline_cluster_label']
TARGET = 'ARR_DEL15'

flights = pd.read_parquet('dataset/merged_flights_fe.parquet')
for col in CAT_FEATURES:
    flights[col] = flights[col].astype('category')

train = flights[flights['FL_DATE'] < '2025-01-01']
test  = flights[flights['FL_DATE'] >= '2025-01-01']

X_train_full = train[FEATURES].copy()
y_train_full = train[TARGET].copy()
X_test       = test[FEATURES].copy()
y_test       = test[TARGET].copy()

print(f"Train: {X_train_full.shape[0]:,}  |  Delay rate: {y_train_full.mean():.4f}")
print(f"Test:  {X_test.shape[0]:,}  |  Delay rate: {y_test.mean():.4f}")

TUNING_SAMPLE = 2_000_000
np.random.seed(42)

delayed_idx = y_train_full[y_train_full == 1].index
ontime_idx  = y_train_full[y_train_full == 0].index
delay_rate  = y_train_full.mean()

sample_idx = np.concatenate([
    np.random.choice(delayed_idx, size=int(TUNING_SAMPLE * delay_rate), replace=False),
    np.random.choice(ontime_idx, size=int(TUNING_SAMPLE * (1 - delay_rate)), replace=False),
])
np.random.shuffle(sample_idx)

X_tune = X_train_full.loc[sample_idx]
y_tune = y_train_full.loc[sample_idx]

print(f"\nTuning subsample: {X_tune.shape[0]:,}  |  Delay rate: {y_tune.mean():.4f}")

del flights, train, test
gc.collect()

print("✓ Data ready")

Train: 13,708,670  |  Delay rate: 0.2069
Test:  4,519,126  |  Delay rate: 0.2297

Tuning subsample: 1,999,999  |  Delay rate: 0.2069
✓ Data ready


In [ ]:
def objective(trial):
    params = {
        'objective':         'binary',
        'metric':            'auc',
        'is_unbalance':      True,
        'verbose':           -1,
        'random_state':      42,
        'n_jobs':            -1,
        'n_estimators':      5000,
        'learning_rate':     trial.suggest_float('learning_rate', 0.005, 0.1, log=True),
        'num_leaves':        trial.suggest_int('num_leaves', 63, 511),
        'max_depth':         trial.suggest_int('max_depth', 6, 15),
        'min_child_samples': trial.suggest_int('min_child_samples', 50, 500),
        'subsample':         trial.suggest_float('subsample', 0.5, 0.9),
        'colsample_bytree':  trial.suggest_float('colsample_bytree', 0.5, 0.9),
        'reg_alpha':         trial.suggest_float('reg_alpha', 1e-3, 10.0, log=True),
        'reg_lambda':        trial.suggest_float('reg_lambda', 1e-3, 10.0, log=True),
        'min_split_gain':    trial.suggest_float('min_split_gain', 0.0, 1.0),
        'max_bin':           trial.suggest_int('max_bin', 127, 511),
    }

    model = lgb.LGBMClassifier(**params)
    model.fit(
        X_tune, y_tune,
        eval_set=[(X_test, y_test)],
        callbacks=[
            lgb.early_stopping(100, verbose=False),
            lgb.log_evaluation(period=0),
        ],
    )

    y_prob = model.predict_proba(X_test)[:, 1]
    auc = roc_auc_score(y_test, y_prob)
    trial.set_user_attr('best_iteration', model.best_iteration_)

    del model
    gc.collect()
    return auc

study = optuna.create_study(
    direction='maximize',
    sampler=optuna.samplers.TPESampler(seed=42),
    pruner=optuna.pruners.MedianPruner(n_startup_trials=10, n_warmup_steps=200),
    study_name='lgbm_flight_delay',
)

def progress_callback(study, trial):
    status = "PRUNED" if trial.state == optuna.trial.TrialState.PRUNED else f"AUC={trial.value:.4f}"
    best = study.best_trial
    print(
        f"Trial {trial.number + 1:>3}/50  |  {status}  |  "
        f"Best: {best.value:.4f} (#{best.number + 1})  |  "
        f"iters: {trial.user_attrs.get('best_iteration', '-')}"
    )

print("Starting Optuna — 50 trials on 2M subsample")
print(f"Baseline AUC: 0.8572 (full data, manual params)")
print("=" * 65)

study.optimize(objective, n_trials=50, callbacks=[progress_callback], gc_after_trial=True)

best = study.best_trial
pruned = len([t for t in study.trials if t.state == optuna.trial.TrialState.PRUNED])
completed = len([t for t in study.trials if t.state == optuna.trial.TrialState.COMPLETE])

print(f"\n{'=' * 65}")
print(f"OPTUNA COMPLETE")
print(f"  Completed: {completed}  |  Pruned: {pruned}")
print(f"  Best AUC (2M tune → full test): {best.value:.4f}")
print(f"  Baseline AUC (full train):      0.8572")
print(f"  Best iteration: {best.user_attrs['best_iteration']}")
print(f"\nBest params:")
for k, v in best.params.items():
    print(f"  {k:.<25s} {v}")

Starting Optuna — 50 trials on 2M subsample
Baseline AUC: 0.8572 (full data, manual params)
Trial   1/50  |  AUC=0.8526  |  Best: 0.8526 (#1)  |  iters: 1120
Trial   2/50  |  AUC=0.8530  |  Best: 0.8530 (#2)  |  iters: 2930
Trial   3/50  |  AUC=0.8516  |  Best: 0.8530 (#2)  |  iters: 913
Trial   4/50  |  AUC=0.8510  |  Best: 0.8530 (#2)  |  iters: 1638
Trial   5/50  |  AUC=0.8513  |  Best: 0.8530 (#2)  |  iters: 5000
Trial   6/50  |  AUC=0.8493  |  Best: 0.8530 (#2)  |  iters: 119
Trial   7/50  |  AUC=0.8519  |  Best: 0.8530 (#2)  |  iters: 903
Trial   8/50  |  AUC=0.8511  |  Best: 0.8530 (#2)  |  iters: 928
Trial   9/50  |  AUC=0.8513  |  Best: 0.8530 (#2)  |  iters: 312
Trial  10/50  |  AUC=0.8525  |  Best: 0.8530 (#2)  |  iters: 2031
Trial  11/50  |  AUC=0.8530  |  Best: 0.8530 (#2)  |  iters: 3625
Trial  12/50  |  AUC=0.8530  |  Best: 0.8530 (#12)  |  iters: 4150
Trial  13/50  |  AUC=0.8525  |  Best: 0.8530 (#12)  |  iters: 1290
Trial  14/50  |  AUC=0.8531  |  Best: 0.8531 (#14)  |